# About this Notebook
This notebook will generate the list of top cuisine types for each country based on Google place types and JET restaurant dim table.
The output of this notebook will be used to subset and aggregate cuisine types of those markets for the lead scoring model.

In [1]:
!pip install fsspec gcsfs seaborn

In [13]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from datetime import datetime
from google.cloud import storage
import re
pd.set_option('display.max_rows', 500)


project_name = 'just-data-sa-dev'
today_date = datetime.today()

client = bigquery.Client(project=project_name, location="EU")


In [3]:

def download_latest_csv(bucket_name, top_level_folder, prefix, contains_pattern, destination_filename):
    """Downloads the latest CSV file from a GCS bucket based on a name prefix.

    Args:
        bucket_name (str): The name of the Google Cloud Storage bucket.
        top_level_folder (str): The top-level folder within the bucket to search.
        prefix (str): The prefix that the desired CSV file names start with.

    Returns:
        str: The local path where the latest CSV file is downloaded, or None if no matching file is found.
    """
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    folder_path = f"{top_level_folder}/"
    blobs = bucket.list_blobs(prefix=folder_path)

    latest_blob = None
    latest_timestamp = None

    for blob in blobs:
        if blob.name.startswith(f"{folder_path}{prefix}") and (contains_pattern in blob.name) and blob.name.endswith("output_data"):
            # Extract timestamp from the folder name:
            match = re.search(r'(\d{14})', blob.name)
            if match:
                try:
                    timestamp_str = match.group(1)
                    blob_timestamp = datetime.strptime(timestamp_str, '%Y%m%d%H%M%S')
                    if latest_timestamp is None or blob_timestamp > latest_timestamp:
                        latest_timestamp = blob_timestamp
                        latest_blob = blob
                except ValueError:
                    print(f"Could not parse timestamp from blob name: {blob.name}")
    
    if latest_blob:
        destination_file_name = f"{destination_filename}.csv"
        latest_blob.download_to_filename(destination_file_name)
        print(f"Downloaded the latest CSV file: {latest_blob.name} to {destination_file_name}")
        return destination_file_name
    else:
        print(f"No CSV files found with prefix '{prefix}' in {bucket_name}/{top_level_folder}/")
        return None
    
    

    

## Download 

In [18]:


# #jet-ml-prod-project-sa-lead-scoring/555228168644/sa-lead-scoring-20250506041002/read-data-2_-1566084851537281024
# bucket_name = "jet-ml-prod-project-sa-lead-scoring"
# top_level_folder = "555228168644"
# prefix = "sa-lead-scoring-"  # Assuming the CSV file name starts with this
# contains_pattern = "read-data-2_"
# destination_filename = "sa-lead-scoring-leads-data"

# downloaded_file_path = download_latest_csv(bucket_name, top_level_folder, prefix, contains_pattern, destination_filename)

# if downloaded_file_path:
#     print(f"Latest CSV file downloaded to: {downloaded_file_path}")

In [19]:
# #gs://jet-ml-prod-project-sa-lead-scoring/555228168644/sa-lead-scoring-20250506041002/read-data_7657287185317494784/output_data 
# bucket_name = "jet-ml-prod-project-sa-lead-scoring"
# top_level_folder = "555228168644"
# prefix = "sa-lead-scoring-"  # Assuming the CSV file name starts with this
# contains_pattern = "read-data_"
# destination_filename = "sa-lead-scoring-train-data"

# downloaded_file_path = download_latest_csv(bucket_name, top_level_folder, prefix, contains_pattern, destination_filename)

# if downloaded_file_path:
#     print(f"Latest CSV file downloaded to: {downloaded_file_path}")

In [20]:
# df_cuisine_leads = pd.read_csv("sa-lead-scoring-leads-data.csv")
# df_cuisine_train = pd.read_csv("sa-lead-scoring-train-data.csv")
# df_cuisine = pd.concat([df_cuisine_leads, df_cuisine_train])

In [21]:
# Load cuisine matching output data
query = """
    SELECT *
    FROM `just-data-warehouse.sales_data_science.cuisine_match_output`
"""
query_job = client.query(query)
df_cuisine = query_job.to_dataframe()

## Group cuisine types into major ones and 'other' type

In [24]:

MINOR_CUISINE_PCT = 0.001
MINOR_CUISINE_COUNT = 100
def group_minor_cuisine(df: pd.DataFrame):
    """If certain cuisine type has less than 100 places and contributes for less than 0.1% of all places in the country, group them under 'other' cuisine."""
    
    # Calculate counts and contributions
    country_counts = df.groupby(["country"], as_index=False).agg(country_total=("google_place_id", "nunique"))
    country_cuisine_counts = df.groupby(["country", "primary_cuisine"], as_index=False).agg(
        place_count=("google_place_id", "nunique")
    )

    cuisine_pct = country_cuisine_counts.merge(country_counts, how="left", on="country")
    cuisine_pct["count_pct"] = cuisine_pct["place_count"] / cuisine_pct["country_total"]

    # Map cuisines to 'other' if they meet criteria
    cuisine_pct["primary_cuisine_new"] = np.where(
        (cuisine_pct["count_pct"] < MINOR_CUISINE_PCT) & (cuisine_pct["place_count"] < MINOR_CUISINE_COUNT),
        "other",
        cuisine_pct["primary_cuisine"],
    )
    
    # Map unclear cuisines to other as well
    cuisine_pct["primary_cuisine_new"] = cuisine_pct["primary_cuisine_new"].replace(
        {"No matching cuisine": 'other', 
         "Unclear cuisine from Google": 'other',
        })
    
    cuisine_pct = cuisine_pct[["country", "primary_cuisine", "primary_cuisine_new"]]
    

    # Update original DataFrame
    df = df.merge(cuisine_pct, how="left", on=["country", "primary_cuisine"])
    
    df["primary_cuisine"] = df["primary_cuisine_new"]
    
    df.drop(columns=["primary_cuisine_new"], inplace=True)

    return df

In [25]:
df_cuisine = group_minor_cuisine(df_cuisine)

In [30]:
# Visualize statistics by the aggregated types
df_total_cuisine = df_cuisine.groupby(['country'], as_index=False).agg(cuisine_count=('google_place_id', 'nunique'))
df_cuisine_agg = df_cuisine.groupby(['country', 'primary_cuisine'], as_index=False).agg({'google_place_id':'nunique'})
df_cuisine_agg = df_cuisine_agg.merge(df_total_cuisine[['country', 'cuisine_count']], how='left', on=['country'])
df_cuisine_agg['count_pct'] = df_cuisine_agg['google_place_id']/df_cuisine_agg['cuisine_count']

In [31]:
check_country = 'IT'
df_cuisine_agg.loc[df_cuisine_agg['country']==check_country].sort_values('count_pct', ascending=False)

,country,primary_cuisine,google_place_id,cuisine_count,count_pct
680,IT,pizza,5189,12237,0.424042
677,IT,other,1486,12237,0.121435
663,IT,fast food,968,12237,0.079104
689,IT,sushi,874,12237,0.071423
649,IT,bar,489,12237,0.039961
685,IT,sandwich,369,12237,0.030154
652,IT,burger,368,12237,0.030073
672,IT,kebab,226,12237,0.018469
670,IT,italian,224,12237,0.018305
681,IT,poke bowl,186,12237,0.015200


## Generate a YAML friendly list of cuisine types

In [50]:
country_list = list(df_cuisine_agg['country'].unique())

In [51]:
for i in country_list:
    subset = df_cuisine_agg.loc[(df_cuisine_agg['country']==i) & (df_cuisine_agg['count_pct'] >= 0.01)]
    print(f"{i}: ")
    for j in subset['primary_cuisine']:
        print(f" - {j.title()}") 

AT: 
 - Bakery
 - Bar
 - Breakfast
 - Cafe
 - Fast Food
 - Grocery
 - Hotel
 - Other
 - Pizza
 - Supermarket
BE: 
 - Bakery
 - Bar
 - Cafe
 - Coffee
 - Fast Food
 - Grocery
 - Hotel
 - Italian
 - Other
 - Pizza
 - Sandwich
 - Snack
 - Supermarket
BG: 
 - Bakery
 - Bar
 - Cafe
 - Coffee
 - Fast Food
 - Grocery
 - Hotel
 - Other
 - Pizza
 - Supermarket
CA: 
 - Bakery
 - Bar
 - Breakfast
 - Cafe
 - Chinese
 - Coffee
 - Dessert
 - Fast Food
 - Grocery
 - Hotel
 - Indian
 - Other
 - Pizza
 - Supermarket
 - Sushi
CH: 
 - Bakery
 - Bar
 - Cafe
 - Coffee
 - Fast Food
 - Grocery
 - Hotel
 - Italian
 - Other
 - Pizza
 - Supermarket
 - Swiss
DE: 
 - Bakery
 - Bar
 - Breakfast
 - Cafe
 - Coffee
 - Fast Food
 - German
 - Grocery
 - Hotel
 - Italian
 - Kebab
 - Other
 - Pizza
 - Snack
 - Supermarket
 - Sushi
DK: 
 - Bakery
 - Bar
 - Burger
 - Cafe
 - Coffee
 - Fast Food
 - Grocery
 - Hotel
 - Other
 - Pizza
 - Supermarket
 - Sushi
ES: 
 - Bakery
 - Bar
 - Burger
 - Cafe
 - Cafeteria
 - Chinese
 - Fa

# Check how many of the current opportunities will be mapped to the new cuisine types

In [32]:
## Get all opportunity snapshot
query = """
    SELECT *
    FROM `just-data-warehouse.sales_analytics_datamarts.global_sales_pipeline`
    WHERE source_system = 'Salesforce - Global' AND account_object_id IS NOT NULL
"""
query_job = client.query(query)
df_account = query_job.to_dataframe()

In [33]:
# Check the cuisine type matches by country
df_join = df_cuisine.merge(df_account[['google_place_id', 'restaurantid']], how='left', on='google_place_id')

In [34]:

df_country_count = df_join.groupby(['country'], as_index=False, dropna=True).agg({'restaurantid': 'nunique'})
df_country_count = df_country_count.rename(columns={'restaurantid':'total_count'})

df_count = df_join.groupby(['country', 'primary_cuisine'], as_index=False, dropna=False).agg({'restaurantid': 'nunique'}).sort_values('restaurantid', ascending=False)
df_count = df_count.merge(df_country_count, how='left', on='country')
df_count['count_pct'] = df_count['restaurantid'] / df_count['total_count']


In [38]:
df_count.loc[df_count['country']=='IT']

,country,primary_cuisine,restaurantid,total_count,count_pct
315,IT,pizza,37,132,0.280303
402,IT,other,19,132,0.143939
434,IT,bar,13,132,0.098485
438,IT,fast food,12,132,0.090909
448,IT,burger,10,132,0.075758
468,IT,poke bowl,6,132,0.045455
483,IT,sandwich,5,132,0.037879
489,IT,kebab,4,132,0.030303
496,IT,sushi,4,132,0.030303
513,IT,hawaiian,3,132,0.022727


In [45]:
# Re-group cuisines that are very rare in the SF opportunities
df_count['primary_cuisine_new'] = np.where((df_count['restaurantid'] < 100) & (df_count['restaurantid'] < 0.001), 'other', df_count['primary_cuisine'])

In [47]:
df_count.loc[df_count['country'] == 'GB']

,country,primary_cuisine,restaurantid,total_count,count_pct,primary_cuisine_new
540,GB,fast food,1,2,0.5,fast food
557,GB,drink,1,2,0.5,drink
587,GB,buffet,0,2,0.0,other
588,GB,brunch,0,2,0.0,other
589,GB,breakfast,0,2,0.0,other
590,GB,brazilian,0,2,0.0,other
591,GB,bbq (barbecue),0,2,0.0,other
592,GB,bar,0,2,0.0,other
593,GB,bangladeshi,0,2,0.0,other
594,GB,bakery,0,2,0.0,other


In [48]:
df_count.groupby(['country', 'primary_cuisine_new'], as_index=False).agg({'count_pct':'sum'})

,country,primary_cuisine_new,count_pct
0,AT,bakery,0.032178
1,AT,bar,0.028575
2,AT,bbq (barbecue),0.002609
3,AT,breakfast,0.017642
4,AT,brunch,0.002609
...,...,...,...
579,SK,snack,0.003125
580,SK,supermarket,0.009375
581,SK,sushi,0.018750
582,SK,tea,0.003125


DE: 
 - Italian
 - Pizza
 - Bakery
 - Bar
 - Coffee
 - Burger
 - Fast Food
 - Kebab
 - Turkish
 - German
 - Cafe
 - Greek
 - Grill
 - Indian
 - Chinese
 - Japanese
 - Vietnamese
 - Sushi
 - American
 - Vegan
 - Thai
PL: 
 - Kebab
 - Pizza
 - Bar
 - Italian
 - Polish
 - Burger
 - Coffee
 - Cafe
 - Sushi
 - Turkish
 - Japanese
 - Bakery
 - Grill
 - American
 - Catering
 - Grocery
 - Fast Food
 - Indian
 - Thai
 - Vegan
 - Vietnamese
 - Dumpling
NL: 
 - Cafe
 - Bar
 - Italian
 - Coffee
 - Chinese
 - Fast Food
 - Bakery
 - Pizza
 - Burger
 - Sushi
 - Grill
 - Turkish
 - Korean
 - Indian
 - Sandwich
 - Indonesian
 - Supermarket
 - Japanese
 - Greek
 - Lunch
 - Vegan
 - Diner
 - Thai
 - Catering
 - Surinamese
 - American
 - Cocktail
 - Bubble Tea
IE: 
 - Pizza
 - Chinese
 - Indian
 - Burger
 - Coffee
 - Fast Food
 - Kebab
 - Cafe
 - Italian
 - American
 - Bar
 - Thai
 - Grill
 - Sandwich
 - Japanese
 - Irish
 - Mexican
 - Grocery
 - Bakery
nan: 
AT: 
 - Bakery
 - Pizza
 - Italian
 - Cafe
 - 

In [ ]:
# Make it formatted for yaml file

